Q-Learning

In [ ]:
import random
import time
from dataclasses import dataclass

# ----------------------------
# 간단한 GridWorld 환경
# ----------------------------
# 상태: (row, col) -> 1차원 상태 id로 변환
# 행동: 0=위, 1=오른쪽, 2=아래, 3=왼쪽
# 보상: 목표 +1, 구멍 -1, 그 외 작은 패널티(옵션), 벽 충돌 시 추가 패널티
# 종료: 목표/구멍/최대 스텝 도달 시 종료

# 행동 번호 상수
UP, RIGHT, DOWN, LEFT = 0, 1, 2, 3
# 각 행동의 이동량
ACTION_DELTAS = {
    UP: (-1, 0),
    RIGHT: (0, 1),
    DOWN: (1, 0),
    LEFT: (0, -1),
}

@dataclass
class GridWorld:
    rows: int = 6
    cols: int = 6
    start: tuple = (0, 0)
    goal: tuple = (5, 5)
    holes: tuple = ((1, 1), (3, 1), (2, 3))
    step_penalty: float = -0.01
    wall_penalty: float = -0.05
    max_steps: int = 50

    # 생성 직후 초기화
    def __post_init__(self):
        self.reset()

    # 좌표 -> 상태 id
    def state_to_id(self, s):
        return s[0] * self.cols + s[1]

    # 상태 id -> 좌표
    def id_to_state(self, sid):
        return (sid // self.cols, sid % self.cols)

    @property
    # 전체 상태 수
    def n_states(self):
        return self.rows * self.cols

    @property
    # 행동 수 (상/우/하/좌)
    def n_actions(self):
        return 4

    # 시작 상태로 리셋
    def reset(self):
        self.pos = self.start
        self.steps = 0
        self.last_hit_wall = False
        return self.state_to_id(self.pos)

    # 한 스텝 진행: 다음 상태/보상/종료 여부 반환
    def step(self, action):

        # 유효한 행동인지 확인
        if action not in ACTION_DELTAS:
            raise ValueError("Invalid action")

        # 행동에 따른 위치 변화 계산
        dr, dc = ACTION_DELTAS[action]
        r, c = self.pos
        r_next = r + dr
        c_next = c + dc

        # 격자 밖으로 못 나가게 처리
        r_clamped = max(0, min(self.rows - 1, r_next))
        c_clamped = max(0, min(self.cols - 1, c_next))

        # 벽 충돌 여부 기록
        self.last_hit_wall = (r_next != r_clamped) or (c_next != c_clamped)

        # 위치 갱신 및 스텝 수 증가
        self.pos = (r_clamped, c_clamped)
        self.steps += 1

        # 종료 조건 체크
        if self.pos in self.holes:
            return self.state_to_id(self.pos), -1.0, True
        if self.pos == self.goal:
            return self.state_to_id(self.pos), +1.0, True
        if self.steps >= self.max_steps:
            return self.state_to_id(self.pos), 0.0, True

        # 기본 보상 설정
        reward = self.step_penalty

        # 벽 충돌 시 패널티 추가
        if self.last_hit_wall:
            reward += self.wall_penalty
        return self.state_to_id(self.pos), reward, False

def epsilon_greedy_action(Q, s, eps):
    # 탐험용 행동 정책: epsilon_greedy로 행동 선택
    if random.random() < eps:  # 0~1사이 난수가 eps 미만이면 탐험 (= eps 확률로 탐험)
        return random.randrange(len(Q[s])) # 임의 행동 선택

    # 탐욕적 행동 선택
    max_q = max(Q[s]) # 상태 s에서 최대 Q값 (현재 상태 s에서 가장 잘 하면 앞으로 얻을 수 있을 것이라고 에이전트가 예상하는 최대 미래 보상 크기 (행동이 아닌 얼마나 좋은 상태인가에 대한 정보))
    best_actions = [a for a, q in enumerate(Q[s]) if q == max_q] # max_q (최대 보상)을 줄 것이라 예상되는 행동들
    return random.choice(best_actions) # 최대 Q값 행동 중 하나 반환


def greedy_action(Q, s):
    # 그리디 행동 선택 (동률이면 랜덤 선택)
    # 애니메이션 용 (학습엔 필요 없음)
    max_q = max(Q[s])
    best_actions = [a for a, q in enumerate(Q[s]) if q == max_q]
    return random.choice(best_actions)

def render_env_with_agent(env):
    """에이전트 위치를 포함해 격자를 출력"""
    border = "+" + "-" * (2 * env.cols - 1) + "+"
    print(border)
    for r in range(env.rows):
        row = []
        for c in range(env.cols):
            if (r, c) == env.pos:
                row.append("A")
            elif (r, c) == env.goal:
                row.append("G")
            elif (r, c) in env.holes:
                row.append("H")
            elif (r, c) == env.start:
                row.append("S")
            else:
                row.append(".")
        print("|" + " ".join(row) + "|")
    print(border)
    if getattr(env, "last_hit_wall", False):
        print("! 벽 충돌")

def animate_greedy_run(env, Q, title, delay=0.05, max_steps=None):
    """그리디 정책으로 한 에피소드 경로를 애니메이션으로 출력"""
    start_time = time.perf_counter()
    env.reset()
    steps = 0
    done = False
    limit = max_steps if max_steps is not None else env.max_steps

    print("\033[H\033[J", end="")
    print(title)
    render_env_with_agent(env)
    time.sleep(delay)

    while not done and steps < limit:
        s = env.state_to_id(env.pos)
        a = greedy_action(Q, s)
        _, _, done = env.step(a)
        steps += 1

        print("\033[H\033[J", end="")  # 화면 지우기
        print(title)
        render_env_with_agent(env)
        time.sleep(delay)
    return time.perf_counter() - start_time

def greedy_action_deterministic(Q, s):
    """그리디 행동 선택 (동률이면 가장 앞의 행동 선택)"""
    max_q = max(Q[s])
    for a, q in enumerate(Q[s]):
        if q == max_q:
            return a
    return 0

def is_adjacent_to_hole(pos, holes):
    """구멍 인접(상하좌우) 칸인지 확인"""
    r, c = pos
    for hr, hc in holes:
        if abs(r - hr) + abs(c - hc) == 1:
            return True
    return False

def evaluate_policy(env, Q, episodes=200, max_steps=None, epsilon=None):
    """정책을 여러 번 실행해 수치로 평가 (epsilon=None이면 그리디)"""
    # 평가 중 난수 소비가 학습에 영향을 주지 않도록 복구
    rng_state = random.getstate()
    random.seed(0)

    success = 0
    hole = 0
    timeout = 0
    total_return = 0.0
    total_steps = 0
    success_steps = 0
    adj_steps = 0
    visited_steps = 0
    limit = max_steps if max_steps is not None else env.max_steps

    for _ in range(episodes):
        s = env.reset()
        done = False
        steps = 0
        ep_return = 0.0

        while not done and steps < limit:
            if epsilon is None:
                a = greedy_action_deterministic(Q, s)
            else:
                a = epsilon_greedy_action(Q, s, epsilon)

            s, r, done = env.step(a)
            ep_return += r
            steps += 1
            visited_steps += 1

            if (env.pos not in env.holes) and is_adjacent_to_hole(env.pos, env.holes):
                adj_steps += 1

        total_return += ep_return
        total_steps += steps

        if env.pos == env.goal:
            success += 1
            success_steps += steps
        elif env.pos in env.holes:
            hole += 1
        else:
            timeout += 1

    random.setstate(rng_state)

    success_rate = success / episodes
    hole_rate = hole / episodes
    timeout_rate = timeout / episodes
    avg_return = total_return / episodes
    avg_steps = total_steps / episodes
    avg_success_steps = (success_steps / success) if success > 0 else 0.0
    adj_rate = (adj_steps / visited_steps) if visited_steps > 0 else 0.0
    return success_rate, hole_rate, timeout_rate, avg_return, avg_steps, avg_success_steps, adj_rate

def train_q_learning(
    env,
    episodes=5000,
    alpha=0.1,
    gamma=0.99,
    eps_start=1.0,
    eps_end=0.05,
    eps_decay=0.999,
    show_animation=True,
    anim_interval=500,
    anim_delay=0.1,
    eval_interval=500,
    eval_episodes=100,
    eval_epsilon=0.05,# 다음 상태, 보상, 에피소드 종료 여부 얻음
    stats=None
):
    # STEP 1. Q 테이블 초기화
    Q = [[0.0 for _ in range(env.n_actions)] for _ in range(env.n_states)]

    eps = eps_start
    perfect_episode = None
    train_start = time.perf_counter()

    anim_time = 0.0
    if show_animation:
        anim_time += animate_greedy_run(env, Q, title="[Q-learning] Episode 0 정책 경로",
                                        delay=anim_delay)

    for ep in range(episodes):
        # 에피소드 시작 상태 초기화
        s = env.reset()
        done = False

        while not done:
            # STEP 2. 행동 정책으로 데이터 수집
            a = epsilon_greedy_action(Q, s, eps)

            # STEP 3. 환경에 행동 적용
            s_next, r, done = env.step(a) # 다음 상태, 보상, 에피소드 종료 여부 얻음

            # STEP 4. TD 업데이트
            Q[s][a] += alpha * (r + gamma * max(Q[s_next]) - Q[s][a])

            s = s_next

        # STEP 5. 탐험률 감소
        eps = max(eps_end, eps * eps_decay)

        # 중간 로그
        if (ep + 1) % 500 == 0:
            print(f"Episode {ep+1}/{episodes} | eps={eps:.3f}")

        # 샘플 효율 체크 (성공률 100%에 도달한 시점)
        if eval_interval > 0 and (ep + 1) % eval_interval == 0 and perfect_episode is None:
            success_rate, _, _, _, _, _, _ = evaluate_policy(
                env, Q, episodes=eval_episodes, epsilon=eval_epsilon
            )
            if success_rate >= 1.0:
                perfect_episode = ep + 1

        # 에피소드 500, 1000, ... 시점에 애니메이션 출력
        if show_animation and ((ep + 1) % anim_interval == 0):
            anim_time += animate_greedy_run(env, Q, title=f"[Q-learning] Episode {ep + 1} 정책 경로",
                                            delay=anim_delay)

    train_end = time.perf_counter()
    train_time = max(0.0, train_end - train_start - anim_time)

    if stats is not None:
        stats["train_time"] = train_time
        stats["anim_time"] = anim_time
        stats["perfect_episode"] = perfect_episode
        stats["eval_interval"] = eval_interval
        stats["eval_episodes"] = eval_episodes
        stats["eval_epsilon"] = eval_epsilon

    return Q

# Q에서 그리디 정책 뽑기
def greedy_policy_from_Q(Q):
    """학습된 Q에서 그리디 정책 생성"""
    policy = []
    for s in range(len(Q)):
        max_q = max(Q[s])
        best_actions = [a for a, q in enumerate(Q[s]) if q == max_q]
        policy.append(random.choice(best_actions))
    return policy

# 정책을 화살표로 출력
def render_policy(env, policy):
    arrows = {UP: "↑", RIGHT: "→", DOWN: "↓", LEFT: "←"}
    for r in range(env.rows):
        row = []
        for c in range(env.cols):
            if (r, c) == env.goal:
                row.append("G")
            elif (r, c) in env.holes:
                row.append("H")
            elif (r, c) == env.start:
                row.append("S")
            else:
                sid = env.state_to_id((r, c))
                row.append(arrows[policy[sid]])
        print(" ".join(row))


if __name__ == "__main__":
    # 재현용 시드 (난수 생성 고정)
    random.seed(0)

    cliff_rows = 7
    cliff_cols = 12
    cliff_holes = tuple((cliff_rows - 1, c) for c in range(1, cliff_cols - 1))
    extra_holes = (
        (1, 2), (1, 7), (2, 4), (2, 9), (3, 1),
        (3, 6), (3, 10), (4, 3), (4, 8), (5, 5),
    )
    env = GridWorld(
        rows=cliff_rows,
        cols=cliff_cols,
        start=(cliff_rows - 1, 0),
        goal=(cliff_rows - 1, cliff_cols - 1),
        holes=cliff_holes + extra_holes
    )
    train_stats = {}
    Q = train_q_learning(
        env,
        episodes=5000,
        alpha=0.1,
        gamma=0.99,
        eps_start=1.0,
        eps_end=0.1,
        eps_decay=0.999,
        stats=train_stats
    )

    print("\n[학습 시간]")
    print(f"학습 소요 시간: {train_stats['train_time']:.2f}초")

    eps_eval = train_stats["eval_epsilon"]
    success_rate, hole_rate, timeout_rate, avg_return, avg_steps, avg_success_steps, adj_rate = evaluate_policy(
        env, Q, episodes=200, epsilon=eps_eval
    )
    print("\n[ε-greedy 정책 평가]")
    print(f"실패율(성공 외 모두): {(1 - success_rate):.2%} (ε={eps_eval:.2f})")
    print(f"성공률: {success_rate:.2%} | 구멍 종료: {hole_rate:.2%} | 제한 종료(최대 {env.max_steps}스텝): {timeout_rate:.2%}")
    print(f"평균 경로 길이(성공 에피소드만): {avg_success_steps:.1f} | 구멍 인접 칸 방문 비율: {adj_rate:.2%}")

    policy = greedy_policy_from_Q(Q)
    print("\nLearned greedy policy:")
    render_policy(env, policy)




[Q-learning] Episode 0 정책 경로
+-----------------------+
|. . . . . . . . . . . .|
|. . H . . . . H . . . .|
|. . . . H . . . . H . .|
|. H . . . . H . . . H .|
|. . . H . . . . H . . .|
|. . . . . H . . . . . .|
|A H H H H H H H H H H G|
+-----------------------+
[Q-learning] Episode 0 정책 경로
+-----------------------+
|. . . . . . . . . . . .|
|. . H . . . . H . . . .|
|. . . . H . . . . H . .|
|. H . . . . H . . . H .|
|. . . H . . . . H . . .|
|. . . . . H . . . . . .|
|A H H H H H H H H H H G|
+-----------------------+
! 벽 충돌
[Q-learning] Episode 0 정책 경로
+-----------------------+
|. . . . . . . . . . . .|
|. . H . . . . H . . . .|
|. . . . H . . . . H . .|
|. H . . . . H . . . H .|
|. . . H . . . . H . . .|
|. . . . . H . . . . . .|
|A H H H H H H H H H H G|
+-----------------------+
! 벽 충돌
[Q-learning] Episode 0 정책 경로
+-----------------------+
|. . . . . . . . . . . .|
|. . H . . . . H . . . .|
|. . . . H . . . . H . .|
|. H . . . . H . . . H .|
|. . . H . . . . H . . .|
|A . . . . H

SARSA 알고리즘

In [ ]:
import random
import time
from dataclasses import dataclass

# ----------------------------
# 간단한 GridWorld 환경
# ----------------------------
# 상태: (row, col) -> 1차원 상태 id로 변환
# 행동: 0=위, 1=오른쪽, 2=아래, 3=왼쪽
# 보상: 목표 +1, 구멍 -1, 그 외 작은 패널티(옵션), 벽 충돌 시 추가 패널티
# 종료: 목표/구멍/최대 스텝 도달 시 종료

# 행동 번호 상수
UP, RIGHT, DOWN, LEFT = 0, 1, 2, 3
# 각 행동의 이동량
ACTION_DELTAS = {
    UP: (-1, 0),
    RIGHT: (0, 1),
    DOWN: (1, 0),
    LEFT: (0, -1),
}

@dataclass
class GridWorld:
    rows: int = 6
    cols: int = 6
    start: tuple = (0, 0)
    goal: tuple = (5, 5)
    holes: tuple = ((1, 1), (3, 1), (2, 3))
    step_penalty: float = -0.01
    wall_penalty: float = -0.05
    max_steps: int = 50

    # 생성 직후 초기화
    def __post_init__(self):
        self.reset()

    # 좌표 -> 상태 id
    def state_to_id(self, s):
        return s[0] * self.cols + s[1]

    # 상태 id -> 좌표
    def id_to_state(self, sid):
        return (sid // self.cols, sid % self.cols)

    @property
    # 전체 상태 수
    def n_states(self):
        return self.rows * self.cols

    @property
    # 행동 수 (상/우/하/좌)
    def n_actions(self):
        return 4

    # 시작 상태로 리셋
    def reset(self):
        self.pos = self.start
        self.steps = 0
        self.last_hit_wall = False
        return self.state_to_id(self.pos)

    # 한 스텝 진행: 다음 상태/보상/종료 여부 반환
    def step(self, action):
        if action not in ACTION_DELTAS:
            raise ValueError("Invalid action")

        dr, dc = ACTION_DELTAS[action]
        r, c = self.pos
        r_next = r + dr
        c_next = c + dc

        # 격자 밖으로 못 나가게 처리
        r_clamped = max(0, min(self.rows - 1, r_next))
        c_clamped = max(0, min(self.cols - 1, c_next))
        self.last_hit_wall = (r_next != r_clamped) or (c_next != c_clamped)
        self.pos = (r_clamped, c_clamped)
        self.steps += 1

        # 종료 조건 체크
        if self.pos in self.holes:
            return self.state_to_id(self.pos), -1.0, True
        if self.pos == self.goal:
            return self.state_to_id(self.pos), +1.0, True
        if self.steps >= self.max_steps:
            return self.state_to_id(self.pos), 0.0, True

        reward = self.step_penalty
        if self.last_hit_wall:
            reward += self.wall_penalty
        return self.state_to_id(self.pos), reward, False


# ----------------------------
# 탭형 SARSA 구현 (온-폴리시)
# ----------------------------
# 입실론-그리디로 행동 선택
def epsilon_greedy_action(Q, s, eps):
    """탐험용 행동 정책: 입실론-그리디"""
    if random.random() < eps:
        return random.randrange(len(Q[s]))
    # 최댓값이 여러 개면 랜덤 선택
    max_q = max(Q[s])
    best_actions = [a for a, q in enumerate(Q[s]) if q == max_q]
    return random.choice(best_actions)

def greedy_action(Q, s):
    """그리디 행동 선택 (동률이면 랜덤 선택)"""
    max_q = max(Q[s])
    best_actions = [a for a, q in enumerate(Q[s]) if q == max_q]
    return random.choice(best_actions)

def render_env_with_agent(env):
    """에이전트 위치를 포함해 격자를 출력"""
    border = "+" + "-" * (2 * env.cols - 1) + "+"
    print(border)
    for r in range(env.rows):
        row = []
        for c in range(env.cols):
            if (r, c) == env.pos:
                row.append("A")
            elif (r, c) == env.goal:
                row.append("G")
            elif (r, c) in env.holes:
                row.append("H")
            elif (r, c) == env.start:
                row.append("S")
            else:
                row.append(".")
        print("|" + " ".join(row) + "|")
    print(border)
    if getattr(env, "last_hit_wall", False):
        print("! 벽 충돌")

def animate_greedy_run(env, Q, title, delay=0.05, max_steps=None):
    """그리디 정책으로 한 에피소드 경로를 애니메이션으로 출력"""
    start_time = time.perf_counter()
    env.reset()
    steps = 0
    done = False
    limit = max_steps if max_steps is not None else env.max_steps

    print("\033[H\033[J", end="")
    print(title)
    render_env_with_agent(env)
    time.sleep(delay)

    while not done and steps < limit:
        s = env.state_to_id(env.pos)
        a = greedy_action(Q, s)
        _, _, done = env.step(a)
        steps += 1

        print("\033[H\033[J", end="")  # 화면 지우기
        print(title)
        render_env_with_agent(env)
        time.sleep(delay)
    return time.perf_counter() - start_time

def greedy_action_deterministic(Q, s):
    """그리디 행동 선택 (동률이면 가장 앞의 행동 선택)"""
    max_q = max(Q[s])
    for a, q in enumerate(Q[s]):
        if q == max_q:
            return a
    return 0

def is_adjacent_to_hole(pos, holes):
    """구멍 인접(상하좌우) 칸인지 확인"""
    r, c = pos
    for hr, hc in holes:
        if abs(r - hr) + abs(c - hc) == 1:
            return True
    return False

def evaluate_policy(env, Q, episodes=200, max_steps=None, epsilon=None):
    """정책을 여러 번 실행해 수치로 평가 (epsilon=None이면 그리디)"""
    # 평가 중 난수 소비가 학습에 영향을 주지 않도록 복구
    rng_state = random.getstate()
    random.seed(0)

    success = 0
    hole = 0
    timeout = 0
    total_return = 0.0
    total_steps = 0
    success_steps = 0
    adj_steps = 0
    visited_steps = 0
    limit = max_steps if max_steps is not None else env.max_steps

    for _ in range(episodes):
        s = env.reset()
        done = False
        steps = 0
        ep_return = 0.0

        while not done and steps < limit:
            if epsilon is None:
                a = greedy_action_deterministic(Q, s)
            else:
                a = epsilon_greedy_action(Q, s, epsilon)

            s, r, done = env.step(a)
            ep_return += r
            steps += 1
            visited_steps += 1

            if (env.pos not in env.holes) and is_adjacent_to_hole(env.pos, env.holes):
                adj_steps += 1

        total_return += ep_return
        total_steps += steps

        if env.pos == env.goal:
            success += 1
            success_steps += steps
        elif env.pos in env.holes:
            hole += 1
        else:
            timeout += 1

    random.setstate(rng_state)

    success_rate = success / episodes
    hole_rate = hole / episodes
    timeout_rate = timeout / episodes
    avg_return = total_return / episodes
    avg_steps = total_steps / episodes
    avg_success_steps = (success_steps / success) if success > 0 else 0.0
    adj_rate = (adj_steps / visited_steps) if visited_steps > 0 else 0.0
    return success_rate, hole_rate, timeout_rate, avg_return, avg_steps, avg_success_steps, adj_rate

def train_sarsa(
    env,
    episodes=5000,
    alpha=0.1,
    gamma=0.99,
    eps_start=1.0,
    eps_end=0.05,
    eps_decay=0.999,
    show_animation=True,
    anim_interval=500,
    anim_delay=0.1,
    eval_interval=500,
    eval_episodes=100,
    eval_epsilon=0.05,
    stats=None
):
    # Q 테이블 초기화
    Q = [[0.0 for _ in range(env.n_actions)] for _ in range(env.n_states)]

    eps = eps_start
    anim_time = 0.0
    perfect_episode = None
    train_start = time.perf_counter()

    if show_animation:
        anim_time += animate_greedy_run(env, Q, title="[SARSA] Episode 0 정책 경로",
                                        delay=anim_delay)

    for ep in range(episodes):
        s = env.reset()
        # [데이터 수집] 입실론-그리디 행동 선택 (행동 정책)
        a = epsilon_greedy_action(Q, s, eps)
        done = False

        while not done:
            s_next, r, done = env.step(a)

            if done:
                # [학습] 종료 상태면 부트스트래핑 없이 업데이트
                td_target = r
                td_error = td_target - Q[s][a]
                Q[s][a] += alpha * td_error
                break

            # SARSA는 "다음에 내가 실제로 할 행동"의 Q값으로 학습하므로
            # 업데이트 전에 다음 상태 s_next에서의 행동 a_next를 먼저 정함
            a_next = epsilon_greedy_action(Q, s_next, eps)

            # [학습] 실제 선택한 행동의 Q값 사용
            td_target = r + gamma * Q[s_next][a_next]
            td_error = td_target - Q[s][a]
            Q[s][a] += alpha * td_error

            s, a = s_next, a_next

        # epsilon 감소
        eps = max(eps_end, eps * eps_decay)

        # 중간 로그
        if (ep + 1) % 500 == 0:
            print(f"Episode {ep+1}/{episodes} | eps={eps:.3f}")

        # 샘플 효율 체크 (성공률 100%에 도달한 시점)
        if eval_interval > 0 and (ep + 1) % eval_interval == 0 and perfect_episode is None:
            success_rate, _, _, _, _, _, _ = evaluate_policy(
                env, Q, episodes=eval_episodes, epsilon=eval_epsilon
            )
            if success_rate >= 1.0:
                perfect_episode = ep + 1

        if show_animation and ((ep + 1) % anim_interval == 0):
            anim_time += animate_greedy_run(env, Q, title=f"[SARSA] Episode {ep + 1} 정책 경로",
                                            delay=anim_delay)

    train_end = time.perf_counter()
    train_time = max(0.0, train_end - train_start - anim_time)

    if stats is not None:
        stats["train_time"] = train_time
        stats["anim_time"] = anim_time
        stats["perfect_episode"] = perfect_episode
        stats["eval_interval"] = eval_interval
        stats["eval_episodes"] = eval_episodes
        stats["eval_epsilon"] = eval_epsilon

    return Q

# Q에서 그리디 정책 뽑기
def greedy_policy_from_Q(Q):
    """학습된 Q에서 그리디 정책 생성"""
    policy = []
    for s in range(len(Q)):
        max_q = max(Q[s])
        best_actions = [a for a, q in enumerate(Q[s]) if q == max_q]
        policy.append(random.choice(best_actions))
    return policy

# 정책을 화살표로 출력
def render_policy(env, policy):
    arrows = {UP: "↑", RIGHT: "→", DOWN: "↓", LEFT: "←"}
    for r in range(env.rows):
        row = []
        for c in range(env.cols):
            if (r, c) == env.goal:
                row.append("G")
            elif (r, c) in env.holes:
                row.append("H")
            elif (r, c) == env.start:
                row.append("S")
            else:
                sid = env.state_to_id((r, c))
                row.append(arrows[policy[sid]])
        print(" ".join(row))


if __name__ == "__main__":
    # 재현용 시드 (난수 생성 고정)
    random.seed(0)

    cliff_rows = 7
    cliff_cols = 12
    cliff_holes = tuple((cliff_rows - 1, c) for c in range(1, cliff_cols - 1))
    extra_holes = (
        (1, 2), (1, 7), (2, 4), (2, 9), (3, 1),
        (3, 6), (3, 10), (4, 3), (4, 8), (5, 5),
    )
    env = GridWorld(
        rows=cliff_rows,
        cols=cliff_cols,
        start=(cliff_rows - 1, 0),
        goal=(cliff_rows - 1, cliff_cols - 1),
        holes=cliff_holes + extra_holes
    )
    train_stats = {}
    Q = train_sarsa(
        env,
        episodes=5000,
        alpha=0.1,
        gamma=0.99,
        eps_start=1.0,
        eps_end=0.1,
        eps_decay=0.999,
        stats=train_stats
    )

    print("\n[학습 시간]")
    print(f"학습 소요 시간: {train_stats['train_time']:.2f}초")

    eps_eval = train_stats["eval_epsilon"]
    success_rate, hole_rate, timeout_rate, avg_return, avg_steps, avg_success_steps, adj_rate = evaluate_policy(
        env, Q, episodes=200, epsilon=eps_eval
    )
    print("\n[ε-greedy 정책 평가]")
    print(f"실패율(성공 외 모두): {(1 - success_rate):.2%} (ε={eps_eval:.2f})")
    print(f"성공률: {success_rate:.2%} | 구멍 종료: {hole_rate:.2%} | 제한 종료(최대 {env.max_steps}스텝): {timeout_rate:.2%}")
    print(f"평균 경로 길이(성공 에피소드만): {avg_success_steps:.1f} | 구멍 인접 칸 방문 비율: {adj_rate:.2%}")

    policy = greedy_policy_from_Q(Q)
    print("\nLearned greedy policy:")
    render_policy(env, policy)



[SARSA] Episode 0 정책 경로
+-----------------------+
|. . . . . . . . . . . .|
|. . H . . . . H . . . .|
|. . . . H . . . . H . .|
|. H . . . . H . . . H .|
|. . . H . . . . H . . .|
|. . . . . H . . . . . .|
|A H H H H H H H H H H G|
+-----------------------+
[SARSA] Episode 0 정책 경로
+-----------------------+
|. . . . . . . . . . . .|
|. . H . . . . H . . . .|
|. . . . H . . . . H . .|
|. H . . . . H . . . H .|
|. . . H . . . . H . . .|
|. . . . . H . . . . . .|
|A H H H H H H H H H H G|
+-----------------------+
! 벽 충돌
[SARSA] Episode 0 정책 경로
+-----------------------+
|. . . . . . . . . . . .|
|. . H . . . . H . . . .|
|. . . . H . . . . H . .|
|. H . . . . H . . . H .|
|. . . H . . . . H . . .|
|. . . . . H . . . . . .|
|A H H H H H H H H H H G|
+-----------------------+
! 벽 충돌
[SARSA] Episode 0 정책 경로
+-----------------------+
|. . . . . . . . . . . .|
|. . H . . . . H . . . .|
|. . . . H . . . . H . .|
|. H . . . . H . . . H .|
|. . . H . . . . H . . .|
|A . . . . H . . . . . .|
|S H H